In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("file_date", "", "File Date")
file_date = dbutils.widgets.get("file_date")

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit
from delta.tables import DeltaTable

In [0]:
movie_genre_df = (
    spark.table("databricks_ws.bronze.movie_genres")
    .filter(col("file_date") == file_date)
    )

In [0]:
movie_genre_selected_df = (
    movie_genre_df
    .select(
        col("movieId").cast("int"),
        col("genreId").cast("int"),
        col("file_date")
    )
)

In [0]:
movie_genre_final_df = (
    add_ingestion_date(movie_genre_selected_df)
    .withColumnRenamed("movieId", "movie_id")
    .withColumnRenamed("genreId", "genre_id")
    .withColumn("environment", lit("Production"))
)

In [0]:
table_name = "databricks_ws.silver.movie_genres"
if spark.catalog.tableExists(table_name):

    deltaTable = DeltaTable.forName(spark, table_name)

    deltaTable.alias("tgt") \
        .merge(
            movie_genre_final_df.alias("src"),
            "tgt.movie_id = src.movie_id AND tgt.genre_id = src.genre_id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
else:
    movie_genre_final_df.write \
        .format("delta") \
        .partitionBy("movie_id") \
        .saveAsTable(table_name)